# NB-07-03 — Sequences, Attention and Transfer Workbench

**Module M07 · Level 300**

Companion to lessons L07-06 to L07-08:

1. An LSTM forecasting a synthetic series, against a naive baseline.
2. A tiny transformer classifying sentiment.
3. Transfer learning: does a pretrained backbone beat training from scratch on 60 labels?
4. An autoencoder used both as a representation learner and as an anomaly detector.

All datasets are synthetic, seeded and CPU-friendly. Numbers here come from real runs; small
differences between runs are expected because of floating-point non-determinism.

In [ ]:
import os, sys, math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Locate the module's code directory whether Jupyter starts here or at the repo root.
for candidate in ("../code", "code", os.path.join("modules", "M07-neural-networks-and-deep-learning", "code")):
    if os.path.isdir(candidate):
        CODE_DIR = os.path.abspath(candidate)
        break
else:
    raise RuntimeError("Run this notebook from the module's notebooks/ directory")
sys.path.insert(0, CODE_DIR)

import deep_data as dd
import nn_numpy
from nn_numpy import MLP, Adam, Momentum, RMSProp, SGD, accuracy, gradient_check, perceptron_rule, train
from torch_utils import (Trainer, SmallCNN, LSTMRegressor, TinyTransformerClassifier, AutoEncoder,
                         count_parameters, freeze, unfreeze, get_device, make_loader, set_seed,
                         load_checkpoint, save_checkpoint)
import transfer

torch.set_num_threads(2)
set_seed(0)
print("code:", CODE_DIR)
print("torch:", torch.__version__, "| device:", get_device())

## 1. LSTM forecasting

The series is a sine wave plus noise plus a slow trend. The model sees 24 past values and predicts
the next one. The honest baseline is "repeat the last value" — always compute it.

In [ ]:
Xs, ys = dd.series_windows(800, window=24, horizon=1, seed=0)
print("windows:", Xs.shape, "targets:", ys.shape)

set_seed(0)
lstm = LSTMRegressor(hidden_size=24)
trainer = Trainer(lstm, criterion=nn.MSELoss())
trainer.fit(make_loader(Xs[:640], ys[:640], 32, seed=0),
            make_loader(Xs[640:], ys[640:], 64, shuffle=False),
            epochs=25, lr=0.01)

test_loader = make_loader(Xs[640:], ys[640:], 64, shuffle=False)
mse, _, _, _ = trainer.evaluate(test_loader)
baseline = np.mean((ys[640:] - Xs[640:, -1, 0]) ** 2)
print("LSTM params:", count_parameters(lstm))
print("LSTM MSE:", round(float(mse), 4), "| repeat-last baseline MSE:", round(float(baseline), 4))

In [ ]:
lstm.eval()
with torch.no_grad():
    preds = lstm(torch.as_tensor(Xs[640:])).squeeze(-1).numpy()
plt.figure(figsize=(9, 3))
plt.plot(ys[640:][:120], label="actual")
plt.plot(preds[:120], label="LSTM", alpha=0.8)
plt.plot(Xs[640:, -1, 0][:120], ":", label="repeat last", alpha=0.7)
plt.legend(); plt.title("one-step-ahead forecasts"); plt.tight_layout(); plt.show()

A one-step forecast on a smooth series is easy; the value is in beating the naive baseline, which it
does by an order of magnitude. Multi-step horizons are where recurrent error accumulation shows up.

## 2. A tiny transformer on sentiment

1,500 synthetic sentences, vocabulary 28, one encoder layer. Compare against the majority-class
baseline, never against nothing.

In [ ]:
texts, labels, vocab = dd.sentiment_data(1500, seed=0)
Xt = dd.encode_sentiment(texts, vocab, max_len=12)
print("vocab:", len(vocab), "encoded:", Xt.shape, "class counts:", np.bincount(labels))

set_seed(0)
transformer = TinyTransformerClassifier(len(vocab) + 1, 12)
tt = Trainer(transformer, patience=6)
tt.fit(make_loader(Xt[:1100], labels[:1100], 64, seed=0),
       make_loader(Xt[1100:], labels[1100:], 256, shuffle=False),
       epochs=12, lr=1e-3)
_, t_acc, _, _ = tt.evaluate(make_loader(Xt[1100:], labels[1100:], 256, shuffle=False))
print("transformer params:", count_parameters(transformer))
print("validation accuracy:", round(t_acc, 3),
      "| majority baseline:", round(max(np.mean(labels[1100:]), 1 - np.mean(labels[1100:])), 3))

On a 1,500-sentence toy task a transformer is overkill; it works, but a GRU or even a
bag-of-words classifier would too. Its advantages appear at scale, which is a claim about data, not
about architecture alone.

## 3. Transfer learning: does the pretrained backbone help?

Pretrain on 3,000 synthetic texture images, then adapt to three-class leaf images with only 60
labelled examples. Three seeds, because one seed proves nothing.

In [ ]:
pretrained = transfer.pretrain_on_textures(epochs=8, seed=0)
print("source task accuracy:", pretrained["pretrain_acc"], "| params:", pretrained["params"])

print(f"{'labels':>6s} {'scratch':>8s} {'frozen':>8s} {'finetune':>9s}")
for n_labels in (60, 150):
    row = []
    for mode in ("scratch", "frozen", "finetune"):
        scores = [transfer.run_experiment(mode, n_labels, seed=s, pretrained=pretrained) for s in (0, 1, 2)]
        row.append(float(np.mean(scores)))
    print(f"{n_labels:6d} {row[0]:8.3f} {row[1]:8.3f} {row[2]:9.3f}")

With 60 labels transfer wins clearly; with 150 labels scratch training has caught up. Transfer is a
data-efficiency tool, and the sweep is the evidence.

## 4. Autoencoder: representation and anomaly detection

Train only on digits 0-7. The reconstruction error becomes an anomaly score for the unseen digits
8 and 9.

In [ ]:
ae, flat, digits, normal = transfer.train_autoencoder(latent=8, epochs=20, seed=0)
errors = ae.reconstruction_error(flat)
threshold = float(np.quantile(errors[normal], 0.95))
flagged = errors > threshold

print("AE params:", count_parameters(ae), "| latent size: 8")
print("mean error normal:", round(float(errors[normal].mean()), 4),
      "| unseen:", round(float(errors[~normal].mean()), 4))
print("threshold:", round(threshold, 4),
      "| flag rate normal:", round(float(flagged[normal].mean()), 3),
      "| recall unseen:", round(float(flagged[~normal].mean()), 3))

plt.figure(figsize=(8, 3))
plt.hist(errors[normal], bins=40, alpha=0.7, label="digits 0-7")
plt.hist(errors[~normal], bins=40, alpha=0.7, label="digits 8-9")
plt.axvline(threshold, color="k", linestyle="--", label="95th percentile")
plt.xlabel("reconstruction error"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

with torch.no_grad():
    codes = ae.encode(torch.as_tensor(flat)).numpy()
print("latent:", codes.shape)
print("1-NN on 8 latent numbers:", round(cross_val_score(KNeighborsClassifier(1), codes, digits, cv=3).mean(), 3))
print("1-NN on 64 raw pixels:    ", round(cross_val_score(KNeighborsClassifier(1), flat, digits, cv=3).mean(), 3))

Eight numbers retain almost all of the neighbour structure of 64 pixels. That is what "a useful
representation" means in practice: a much smaller vector that still supports the downstream task.

## 5. Exercise

1. Retrain the LSTM with `horizon=3` and compare the MSE against a repeat-last baseline for 3 steps.
2. Halve the transformer's training set. Does validation accuracy fall faster than the GRU's?
3. Run the transfer sweep with digit pretraining instead of textures. Is it negative transfer?
4. Train the autoencoder with `latent=2`, plot the codes, and name two digits that overlap.

**References:** L07-06, L07-07, L07-08. Code: `code/torch_utils.py`, `code/transfer.py`.